In [1]:
# Parameters
summary_config = {"run_run_comparison": False, "run_RTP_summary": False, "run_validation": True, "run_network_validation": True, "summary_list": {"RTP-summary-notebook": ["RTP_index", "RTP_congestion", "RTP_topsheet", "RTP_MIC", "RTP_person", "RTP_household", "RTP_access", "RTP_costs", "RTP_walk_bike", "RTP_emissions", "RTP_mode_share", "RTP_freight", "RTP_transit"], "activitysim-validation-notebook": ["work_from_home", "auto_ownership", "telecommute_frequency", "free_parking", "cdap", "intermediate_stop_frequency", "trip_purpose", "trip_destination_choice", "school_location", "work_location", "mandatory_tour_frequency", "mandatory_tour_scheduling", "non_mandatory_tour_frequency", "non_mandatory_tour_destination_choice", "non_mandatory_tour_scheduling", "joint_tour_frequency", "joint_tour_composition", "atwork_subtours_frequency", "atwork_subtours_destination_choice", "atwork_subtours_scheduling", "atwork_subtour_mode", "tour_mode_choice", "trip_mode_choice"], "daysim-validation-notebook": ["all_tour_mode", "all_trip_mode", "auto_ownership", "day_pattern", "escort_tour_mode", "escort_trip_mode", "households", "intermediate_stop_generation", "other_home_based_tour_mode", "other_home_based_trip_mode", "persons", "school_location", "school_tour_mode", "school_trip_mode", "telecommute", "time_choice", "tours", "tour_destination", "transit_pass_ownership", "trips", "trip_destination", "workbased_subtour_generation", "workbased_subtour_mode", "work_location", "work_tour_mode", "work_trip_mode"], "network-validation-notebook": ["JBLM", "supplementals", "transit_validation", "traffic_validation", "bike_validation", "link_analysis"], "run-comparison-notebook": ["topsheet", "population", "parking", "vmt", "transit"]}, "p_output_dir": "outputs/summary", "output_folder": "outputs", "survey_folder": "inputs/base_year/survey", "uncloned_folder": "uncloned", "sc_run_name": "current run", "sc_run_path": "../../../../", "survey_directories": {"survey": "../../../../inputs/base_year/survey"}, "comparison_runs_list": {"2050 new transit, old network": "\\\\modelstation3\\c$\\Workspace\\sc_new_2050_transit\\soundcast", "2050 urbansim": "\\\\modelstation2\\c$\\Workspace\\sc_2050_urbansim2_07_30_25"}, "county_map": {"33": "King", "35": "Kitsap", "53": "Pierce", "61": "Snohomish"}, "uc_list": ["@sov_inc1", "@sov_inc2", "@sov_inc3", "@hov2_inc1", "@hov2_inc2", "@hov2_inc3", "@hov3_inc1", "@hov3_inc2", "@hov3_inc3", "@av_sov_inc1", "@av_sov_inc2", "@av_sov_inc3", "@av_hov2_inc1", "@av_hov2_inc2", "@av_hov2_inc3", "@av_hov3_inc1", "@av_hov3_inc2", "@av_hov3_inc3", "@tnc_inc1", "@tnc_inc2", "@tnc_inc3", "@mveh", "@hveh", "@bveh"], "agency_lookup": {"1": "King County Metro", "2": "Pierce Transit", "3": "Community Transit", "4": "Kitsap Transit", "5": "Washington Ferries", "6": "Sound Transit", "7": "Everett Transit"}, "emissions_scenario": "standard", "tot_veh_model_base_year": 3185281, "speed_bins": [-999999.0, 2.5, 7.5, 12.5, 17.5, 22.5, 27.5, 32.5, 37.5, 42.5, 47.5, 52.5, 57.5, 62.5, 67.5, 72.5, 999999.0], "fac_type_lookup": {"0": 0, "1": 4, "2": 4, "3": 5, "4": 5, "5": 5, "6": 3, "7": 5, "8": 0}, "tod_lookup": {"5to9": 5, "9to15": 9, "15to18": 15, "18to20": 18, "20to5": 20}, "summer_list": [87], "special_route_lookup": {"1671": "A-Line Rapid Ride", "1672": "B-Line Rapid Ride", "1673": "C-Line Rapid Ride", "1674": "D-Line Rapid Ride", "1675": "E-Line Rapid Ride", "1677": "H-Line Rapid Ride", "4950": "Central Link", "6995": "Tacoma Link", "6998": "Sounder South", "6999": "Sounder North", "3701": "Swift Blue Line", "3702": "Swift Green Line"}}
input_config = {"debug_skims_and_paths": False, "model_year": "2023", "base_year": "2023", "landuse_inputs": "23_on_23_v3", "network_inputs": "base_year_2023_final", "db_name": "soundcast_inputs_2023.db", "soundcast_inputs_dir": "R:/e2projects_two/SoundCast/Inputs/rtp_2026_2050", "abm_model": "daysim", "run_accessibility_calcs": False, "run_setup_emme_project_folders": False, "run_setup_emme_bank_folders": False, "run_copy_scenario_inputs": False, "run_import_networks": False, "run_skims_and_paths_free_flow": False, "run_skims_and_paths": False, "run_truck_model": False, "run_supplemental_trips": False, "run_daysim": False, "run_summaries": True, "include_av": False, "include_tnc": True, "tnc_av": False, "include_tnc_to_transit": False, "include_knr_to_transit": False, "include_delivery": False, "include_telecommute": True, "run_integrated": False, "should_build_shadow_price": False, "delete_banks": False, "include_tnc_emissions": True, "add_distance_pricing": False, "distance_rate_dict": {"am": 13.5, "md": 8.5, "pm": 13.5, "ev": 8.5, "ni": 8.5}}


In [2]:
# import toml
# from pathlib import Path
# config_path = Path("../../../../configuration")
# input_config = toml.load(config_path/ "input_configuration.toml")
# summary_config = toml.load(config_path/ "configuration/summary_configuration.toml")

In [3]:
import polars as pl
import plotly.express as px
import os, sys

module_path = os.path.abspath(os.path.join('../../../..')) # or the path to your source code
sys.path.insert(0, module_path)
import util

In [4]:
person = util.get_person_data(summary_config)
tour = util.get_tour_data(summary_config)

## number of tours per person

In [5]:
df_tour = tour.join(person, on=['hhno','pno','source'], how='left')

In [6]:
df_plot = df_tour.group_by('source').agg([
    pl.sum('toexpfac').alias('toexpfac_sum')
]).join(
    person.group_by('source').agg([
        pl.sum('psexpfac').alias('psexpfac_sum')
    ]),
    on='source',
    how='left'
).with_columns(
    (pl.col('toexpfac_sum') / pl.col('psexpfac_sum')).alias('average tours'),
    pl.lit('').alias('person')
)

fig = px.bar(df_plot, x="person", y="average tours", color="source",
             barmode="group", title="number of tours per person")
fig.update_layout(height=400, width=400, font=dict(size=11),
                  xaxis=dict(dtick=1, categoryorder='category ascending'),
                  yaxis=dict(tickformat=".3"))
fig.show()

## percent of tours by purpose

In [7]:
df_plot = df_tour.group_by(['source', 'pdpurp_label']).agg([
    pl.sum('toexpfac').alias('toexpfac_sum')
]).with_columns(
    (pl.col('toexpfac_sum') / pl.col('toexpfac_sum').sum().over('source')).alias('percentage')
)

df_plot_ct = df_tour.group_by(['source', 'pdpurp_label']).agg([
    pl.count('toexpfac').alias('sample count')
])

df_plot = df_plot.join(df_plot_ct, on=['source', 'pdpurp_label'], how='left').sort(['source', 'pdpurp_label'])

fig = px.bar(df_plot, x="pdpurp_label", y="percentage", color="source",
             barmode="group", hover_data=['sample count'], title="tour purpose")
fig.update_layout(height=400, width=700, font=dict(size=11),
                  yaxis=dict(tickformat=".2%"))
fig.show()

## number of tours per person by segment

In [8]:
def plot_segment(df, tour_group_var, person_group_var, title_name):
    fig = px.bar(df, x=tour_group_var[1], y="average tour", color="source",
                 barmode="group", title=title_name)
    fig.update_layout(height=400, width=700, font=dict(size=11),
                      yaxis=dict(tickformat=".2"))
    fig.show()

df_plot = df_tour.group_by(['source', 'pdpurp_label']).agg([
    pl.sum('toexpfac').alias('toexpfac_sum')
]).join(
    person.group_by(['source']).agg([
        pl.sum('psexpfac').alias('psexpfac_sum')
    ]),
    on='source',
    how='left'
).with_columns(
    (pl.col('toexpfac_sum') / pl.col('psexpfac_sum')).alias('average tour')
).sort(['source', 'pdpurp_label'])

plot_segment(df_plot, tour_group_var=['source', 'pdpurp_label'], person_group_var=['source'],
             title_name="number of tours per person by tour purpose")


In [9]:
# plot_segment(tour_group_var=['source','pptyp_label'],person_group_var=['source','pptyp_label'],
#              title_name="number of tours per person by person type")

df_plot = df_tour.group_by(['source', 'pptyp_label']).agg([
    pl.sum('toexpfac').alias('toexpfac_sum')
]).join(
    person.group_by(['source', 'pptyp_label']).agg([
        pl.sum('psexpfac').alias('psexpfac_sum')
    ]),
    on=['source', 'pptyp_label'],
    how='left'
).with_columns(
    (pl.col('toexpfac_sum') / pl.col('psexpfac_sum')).alias('average tour')
).sort(['source', 'pptyp_label'])

plot_segment(df_plot, tour_group_var=['source', 'pptyp_label'], person_group_var=['source', 'pptyp_label'],
             title_name="number of tours per person by person type")

In [10]:
wk_tour = df_tour.filter(pl.col('pdpurp') == 1)

df_plot = wk_tour.group_by(['source', 'pptyp_label']).agg([
    pl.sum('toexpfac').alias('toexpfac_sum')
]).join(
    person.group_by(['source', 'pptyp_label']).agg([
        pl.sum('psexpfac').alias('psexpfac_sum')
    ]),
    on=['source', 'pptyp_label'],
    how='left'
).with_columns(
    (pl.col('toexpfac_sum') / pl.col('psexpfac_sum')).alias('average tour')
).sort(['source', 'pptyp_label'])

fig = px.bar(df_plot, x='pptyp_label', y="average tour", color="source",
             barmode="group", title="number of work tours per person by person type")
fig.update_layout(height=400, width=700, font=dict(size=11),
                  yaxis=dict(tickformat=".2"))
fig.show()

### Tour by Purpose and Person Type

In [11]:

def plot_by_pptyp(df_tour, person_type):
    df_plot = df_tour.filter(
        pl.col('pptyp') == int(person_type)
    ).group_by(['source', 'pdpurp_label']).agg([
        pl.sum('toexpfac').alias('toexpfac_sum')
    ]).join(
        person.filter(pl.col('pptyp') == int(person_type)).group_by(['source']).agg([
            pl.sum('psexpfac').alias('psexpfac_sum')
        ]),
        on=['source'],
        how='left'
    ).with_columns(
        (pl.col('toexpfac_sum') / pl.col('psexpfac_sum')).alias('average tour')
    ).sort(['source', 'pdpurp_label'])

    plot_segment(df_plot, tour_group_var=['source', 'pdpurp_label'], person_group_var=['source'],
                 title_name="number of tours per person for person type " + str(person_type))


In [12]:
plot_by_pptyp(df_tour, '1')

In [13]:
plot_by_pptyp(df_tour, '2')

In [14]:
plot_by_pptyp(df_tour, '3')

In [15]:
plot_by_pptyp(df_tour, '4')

In [16]:
plot_by_pptyp(df_tour, '5')

In [17]:
plot_by_pptyp(df_tour, '6')

In [18]:
plot_by_pptyp(df_tour, '7')

In [19]:
plot_by_pptyp(df_tour, '8')